# Figure 6 Lyapunov-Style Stability Compilation

Cache-first notebook for testing whether distance-dependent spike-rate trajectories show Lyapunov-style divergence structure. The workflow is intentionally run **per DIV** so expensive or failed DIVs do not invalidate previously computed results.

The implementation avoids the main inefficiencies in the legacy `StabilityAnalyzer` path:

- delay embeddings are built once per recording/electrode instead of once per pair;
- candidate electrode pairs are sampled per distance bin before expensive nearest-neighbor work;
- time points inside selected activity intervals are sampled before pairwise distance matrices are built;
- derived result tables are cached per DIV, not raw traces or dense pair intermediates.

In [ ]:
%matplotlib inline

from __future__ import annotations

from pathlib import Path
import ast
import hashlib
import json
import pickle
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.spatial.distance import pdist, squareform
from scipy.stats import ttest_ind
from numpy.lib.stride_tricks import sliding_window_view

repo_root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ephax import PrepConfig, RestingActivityDataset
from ephax.metrics.burst import (
    build_highres_traces,
    build_participation_activity_state,
    build_population_ifr,
    detect_high_activity_epochs,
    detect_participation_burst_epochs,
)
from ephax.plotting import PAPER_COLORS, apply_paper_style, compose_figure, export_figure, figure, panel

apply_paper_style()
plt.rcParams["figure.dpi"] = 170

## Configuration

Set `FORCE_RECOMPUTE_DIVS` to the DIVs that should be rebuilt. Leaving it empty makes the notebook load compatible per-DIV caches when available.

In [ ]:
DATASET = "stim_removal_null"
RUN_ALL_WELLS = True
RUN_ALL_DIVS = True
WELL = 0
DIV = 12
AGGREGATE_WELLS = [0, 1, 2, 3, 4, 5]
AGGREGATE_DIVS = [12, 14, 21, 22, 23]
REQUESTED_WELLS = AGGREGATE_WELLS if RUN_ALL_WELLS else [WELL]
REQUESTED_DIVS = AGGREGATE_DIVS if RUN_ALL_DIVS else [DIV]

START_SEC = 0.0
END_SEC = 300.0
MIN_AMP = 0.0
TOP_START = 0
TOP_STOP = 1000

# Activity-state detection, aligned with the burst/activity-distance notebooks.
IFR_GRID_HZ = 50.0
SMOOTH_SIGMA_SEC = 0.15
HIGH_ACTIVITY_MAD_SCALE = 3.0
HIGH_ACTIVITY_MIN_DURATION_MS = 30.0
HIGH_ACTIVITY_MAX_GAP_BINS = 0
HIGHRES_BIN_MS = 1.0
HIGHRES_SMOOTH_SIGMA_MS = 3.0
NETWORK_BIN_MS = 10.0
NETWORK_MIN_PARTICIPATION_FRACTION = 0.05
NETWORK_MIN_DURATION_MS = 20.0
LYAP_ACTIVITY_SCOPE = "burst"  # "burst", "high_activity_non_burst", or "high_activity_all"

# Lyapunov-style computation. Keep these moderate for first pass; increase max pairs per bin later.
LYAP_BIN_WIDTH_S = 1.0 / IFR_GRID_HZ
LYAP_EMBED_DIM = 4
LYAP_DELAY_BINS = 1
LYAP_MAX_TIME_POINTS = 250
LYAP_MAX_PAIRS_PER_DISTANCE_BIN = 250
LYAP_MIN_DISTANCE_UM = 50.0
LYAP_MAX_DISTANCE_UM = 3500.0
LYAP_DISTANCE_BIN_UM = 200.0
LYAP_MAX_DT_BINS = 10
LYAP_EPSILON_QUANTILE = 0.08
LYAP_MIN_NEIGHBORS = 5
LYAP_MAX_NEIGHBORS = 12
LYAP_BOOTSTRAP_REPS = 1000
RANDOM_SEED = 0

FORCE_RECOMPUTE_DIVS = []
SAVE_FIGURE = True
OUTPUT_DIR = repo_root / "outputs" / "figure6_lyapunov"
TABLE_DIR = OUTPUT_DIR / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

STIM_DATA_ROOT = repo_root / "ephax" / "data" / "stimRemovalNull"
STIM_DATA_FILENAME_TEMPLATE = "DIV{div}_240703_data_well{well}_exp_data.npz"

## Cache Helpers and Recording Specs

In [ ]:
def stable_digest(obj, n=12):
    payload = json.dumps(obj, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:n]


def lyap_config_dict():
    return {
        "dataset": DATASET,
        "wells": list(map(int, REQUESTED_WELLS)),
        "divs": list(map(int, REQUESTED_DIVS)),
        "start_sec": START_SEC,
        "end_sec": END_SEC,
        "min_amp": MIN_AMP,
        "top_start": TOP_START,
        "top_stop": TOP_STOP,
        "activity_scope": LYAP_ACTIVITY_SCOPE,
        "ifr_grid_hz": IFR_GRID_HZ,
        "smooth_sigma_sec": SMOOTH_SIGMA_SEC,
        "high_activity_mad_scale": HIGH_ACTIVITY_MAD_SCALE,
        "high_activity_min_duration_ms": HIGH_ACTIVITY_MIN_DURATION_MS,
        "high_activity_max_gap_bins": HIGH_ACTIVITY_MAX_GAP_BINS,
        "highres_bin_ms": HIGHRES_BIN_MS,
        "network_bin_ms": NETWORK_BIN_MS,
        "network_min_participation_fraction": NETWORK_MIN_PARTICIPATION_FRACTION,
        "network_min_duration_ms": NETWORK_MIN_DURATION_MS,
        "embed_dim": LYAP_EMBED_DIM,
        "delay_bins": LYAP_DELAY_BINS,
        "max_time_points": LYAP_MAX_TIME_POINTS,
        "max_pairs_per_bin": LYAP_MAX_PAIRS_PER_DISTANCE_BIN,
        "distance_bin_um": LYAP_DISTANCE_BIN_UM,
        "min_distance_um": LYAP_MIN_DISTANCE_UM,
        "max_distance_um": LYAP_MAX_DISTANCE_UM,
        "max_dt_bins": LYAP_MAX_DT_BINS,
        "epsilon_quantile": LYAP_EPSILON_QUANTILE,
        "min_neighbors": LYAP_MIN_NEIGHBORS,
        "max_neighbors": LYAP_MAX_NEIGHBORS,
        "bootstrap_reps": LYAP_BOOTSTRAP_REPS,
        "seed": RANDOM_SEED,
    }


CONFIG_DIGEST = stable_digest(lyap_config_dict())


def div_cache_paths(div):
    stem = f"lyapunov_div{int(div)}_{CONFIG_DIGEST}"
    return {
        "summary": TABLE_DIR / f"{stem}_distance_summary.csv.gz",
        "pairs": TABLE_DIR / f"{stem}_pair_samples.csv.gz",
        "recordings": TABLE_DIR / f"{stem}_recording_summary.csv.gz",
        "payload": OUTPUT_DIR / f"{stem}_payload.pkl",
    }


def stim_recording_path(well, div):
    return STIM_DATA_ROOT / f"well{int(well)}" / STIM_DATA_FILENAME_TEMPLATE.format(div=int(div), well=int(well))


def build_recording_spec(well, div):
    if DATASET != "stim_removal_null":
        raise ValueError("Figure 6 currently expects DATASET='stim_removal_null'.")
    path = stim_recording_path(well, div)
    if not path.exists():
        raise FileNotFoundError(path)
    return {
        "dataset": DATASET,
        "source": "npz",
        "path": path,
        "well": int(well),
        "div": int(div),
        "recording_id": f"stimRemovalNull_well{int(well)}_DIV{int(div)}",
        "source_file": path.name,
    }


def iter_recording_specs_for_div(div):
    for well in REQUESTED_WELLS:
        try:
            yield build_recording_spec(well, div)
        except FileNotFoundError as exc:
            print(f"Skipping missing recording: {exc}")


def load_recording_from_spec(spec):
    spec["resolved_start_sec"] = float(START_SEC)
    spec["resolved_end_sec"] = float(END_SEC)
    file_info = [(str(spec["path"]), float(START_SEC), float(END_SEC), int(spec["well"]))]
    return RestingActivityDataset.from_file_info(
        file_info,
        source="npz",
        min_amp=MIN_AMP,
    )


## Interval and Trajectory Helpers

The Lyapunov calculation uses log-smoothed spike-rate trajectories on the same time grid for all selected electrodes. Activity intervals are used as a mask on this trajectory, so removed gaps are not treated as continuous dynamics.

In [ ]:
def empty_intervals():
    return np.empty((0, 2), dtype=float)


def normalize_intervals(intervals):
    arr = np.asarray(intervals, dtype=float)
    if arr.size == 0:
        return empty_intervals()
    arr = arr.reshape(-1, 2)
    arr = arr[np.isfinite(arr).all(axis=1)]
    arr = arr[arr[:, 1] > arr[:, 0]]
    if arr.size == 0:
        return empty_intervals()
    arr = arr[np.argsort(arr[:, 0])]
    merged = []
    for start, stop in arr:
        if not merged or start > merged[-1][1]:
            merged.append([float(start), float(stop)])
        else:
            merged[-1][1] = max(merged[-1][1], float(stop))
    return np.asarray(merged, dtype=float)


def subtract_intervals(base, remove):
    base = normalize_intervals(base)
    remove = normalize_intervals(remove)
    if base.size == 0 or remove.size == 0:
        return base.copy()
    pieces = []
    for start, stop in base:
        cursor = start
        for r0, r1 in remove:
            if r1 <= cursor or r0 >= stop:
                continue
            if r0 > cursor:
                pieces.append([cursor, min(r0, stop)])
            cursor = max(cursor, r1)
            if cursor >= stop:
                break
        if cursor < stop:
            pieces.append([cursor, stop])
    return normalize_intervals(pieces)


def choose_activity_intervals(high_epochs, burst_epochs, scope):
    high_epochs = normalize_intervals(high_epochs)
    burst_epochs = normalize_intervals(burst_epochs)
    if scope == "high_activity_all":
        return high_epochs
    if scope == "high_activity_non_burst":
        return subtract_intervals(high_epochs, burst_epochs)
    if scope == "burst":
        return burst_epochs
    raise ValueError("LYAP_ACTIVITY_SCOPE must be 'burst', 'high_activity_non_burst', or 'high_activity_all'.")


def interval_mask(time_grid, intervals):
    mask = np.zeros(time_grid.shape, dtype=bool)
    for start, stop in normalize_intervals(intervals):
        mask |= (time_grid >= start) & (time_grid <= stop)
    return mask


def detect_recording_activity(spec):
    try:
        ds = load_recording_from_spec(spec)
        rec = ds.recordings[0]
        prep_cfg = PrepConfig(mode="top", top_start=TOP_START, top_stop=TOP_STOP, verbose=False)
        refs = ds.select_ref_electrodes(prep_cfg)[0]
        if len(refs) == 0:
            raise ValueError("no selected electrodes")
        population = build_population_ifr(rec, refs, grid_hz=IFR_GRID_HZ, smooth_sigma_sec=SMOOTH_SIGMA_SEC)
        high_epochs, high_info = detect_high_activity_epochs(
            population.time_grid,
            population.mean_ifr_smooth,
            mad_scale=HIGH_ACTIVITY_MAD_SCALE,
            min_duration_ms=HIGH_ACTIVITY_MIN_DURATION_MS,
            max_gap_bins=HIGH_ACTIVITY_MAX_GAP_BINS,
        )
        try:
            highres = build_highres_traces(rec, refs, bin_ms=HIGHRES_BIN_MS, smooth_sigma_ms=HIGHRES_SMOOTH_SIGMA_MS)
            participation = build_participation_activity_state(highres, aggregation_ms=NETWORK_BIN_MS)
            burst_epochs = detect_participation_burst_epochs(
                participation,
                high_epochs,
                min_participation_fraction=NETWORK_MIN_PARTICIPATION_FRACTION,
                min_duration_ms=NETWORK_MIN_DURATION_MS,
            )
        except ValueError:
            burst_epochs = empty_intervals()
        selected_intervals = choose_activity_intervals(high_epochs, burst_epochs, LYAP_ACTIVITY_SCOPE)
        selected_intervals = normalize_intervals(selected_intervals)
        return {
            "ok": True,
            "spec": spec,
            "dataset": ds,
            "recording": rec,
            "refs": np.asarray(refs, dtype=int),
            "population": population,
            "high_epochs": high_epochs,
            "burst_epochs": burst_epochs,
            "selected_intervals": selected_intervals,
            "skip_reason": "",
        }
    except Exception as exc:
        return {
            "ok": False,
            "spec": spec,
            "dataset": None,
            "recording": None,
            "refs": np.asarray([], dtype=int),
            "population": None,
            "high_epochs": empty_intervals(),
            "burst_epochs": empty_intervals(),
            "selected_intervals": empty_intervals(),
            "skip_reason": str(exc),
        }


def build_rate_matrix(recording, electrodes, time_grid):
    electrodes = np.asarray(electrodes, dtype=int)
    dt = float(np.nanmedian(np.diff(time_grid))) if len(time_grid) > 1 else LYAP_BIN_WIDTH_S
    edges = np.r_[time_grid - 0.5 * dt, time_grid[-1] + 0.5 * dt]
    spike_times = np.asarray(recording.spikes["time"], dtype=float)
    spike_electrodes = np.asarray(recording.spikes["electrode"], dtype=int)
    order = np.argsort(electrodes)
    sorted_electrodes = electrodes[order]
    positions = np.searchsorted(sorted_electrodes, spike_electrodes)
    valid_pos = positions < sorted_electrodes.size
    matched = np.zeros(spike_electrodes.shape, dtype=bool)
    matched[valid_pos] = sorted_electrodes[positions[valid_pos]] == spike_electrodes[valid_pos]
    matched &= (spike_times >= edges[0]) & (spike_times <= edges[-1])
    counts = np.zeros((len(electrodes), len(time_grid)), dtype=np.float32)
    if np.any(matched):
        rows = order[positions[matched]]
        cols = np.searchsorted(edges, spike_times[matched], side="right") - 1
        keep = (cols >= 0) & (cols < len(time_grid))
        np.add.at(counts, (rows[keep], cols[keep]), 1.0)
    rates = counts / np.float32(dt)
    sigma_bins = float(SMOOTH_SIGMA_SEC) / max(dt, 1e-12)
    rates = gaussian_filter1d(rates, sigma=sigma_bins, axis=1, mode="nearest", output=np.float32)
    return np.log10(rates + np.float32(1e-4))


def make_delay_embedding_matrix(rate_matrix, embed_dim=LYAP_EMBED_DIM, delay_bins=LYAP_DELAY_BINS):
    # Input channels x time. Output time x channels x embed_dim.
    channels, n_time = rate_matrix.shape
    span = 1 + (embed_dim - 1) * delay_bins
    if n_time < span:
        return np.empty((0, channels, embed_dim), dtype=np.float32)
    windows = sliding_window_view(rate_matrix, window_shape=span, axis=1)  # channels x n_embed x span
    sampled = windows[:, :, ::delay_bins]
    return np.moveaxis(sampled, 1, 0).astype(np.float32, copy=False)

## Pairwise Lyapunov Computation

This is a sampled pair statistic, not a full all-pairs scan. The expensive operation is the neighbor-distance matrix inside each pair; therefore we cap both time points and pairs per distance bin.

In [ ]:
def compute_pair_lyapunov_from_embeddings(embed_a, embed_b, rng):
    n = min(len(embed_a), len(embed_b))
    if n <= LYAP_MAX_DT_BINS + 5:
        return np.nan
    if n > LYAP_MAX_TIME_POINTS:
        sample_n = min(int(LYAP_MAX_TIME_POINTS), max(0, n - LYAP_MAX_DT_BINS))
        if sample_n <= LYAP_MAX_DT_BINS + 5:
            return np.nan
        idx = np.sort(rng.choice(n - LYAP_MAX_DT_BINS, size=sample_n, replace=False))
        embed_a = embed_a[idx]
        embed_b = embed_b[idx]
        n = len(idx)
    joint = np.hstack([embed_a, embed_b]).astype(np.float64, copy=False)
    std = np.nanstd(joint, axis=0)
    std[std == 0] = 1.0
    joint = (joint - np.nanmean(joint, axis=0)) / std
    dist = squareform(pdist(joint, metric="euclidean"))
    finite = dist[np.isfinite(dist) & (dist > 0)]
    if finite.size == 0:
        return np.nan
    epsilon = float(np.nanquantile(finite, LYAP_EPSILON_QUANTILE))
    if not np.isfinite(epsilon) or epsilon <= 0:
        return np.nan
    values = []
    for i in range(n - LYAP_MAX_DT_BINS):
        neigh = np.where((dist[i] > 0) & (dist[i] <= epsilon))[0]
        neigh = neigh[neigh < n - LYAP_MAX_DT_BINS]
        if neigh.size < LYAP_MIN_NEIGHBORS:
            continue
        if neigh.size > LYAP_MAX_NEIGHBORS:
            neigh = neigh[np.argsort(dist[i, neigh])[:LYAP_MAX_NEIGHBORS]]
        initial = dist[i, neigh]
        for dt in range(1, LYAP_MAX_DT_BINS + 1):
            future = dist[i + dt, neigh + dt]
            with np.errstate(divide="ignore", invalid="ignore"):
                log_ratio = np.log((future + 1e-12) / (initial + 1e-12)) / dt
            log_ratio = log_ratio[np.isfinite(log_ratio)]
            if log_ratio.size:
                values.append(float(np.mean(log_ratio)))
    if not values:
        return np.nan
    return float(np.mean(values))


def electrode_coords(recording):
    layout = pd.DataFrame(recording.layout)
    return layout.set_index("electrode")[["x", "y"]].astype(float)


def sample_pairs_by_distance(electrodes, coords, rng):
    electrodes = np.asarray(electrodes, dtype=int)
    coords_sel = coords.loc[electrodes, ["x", "y"]].to_numpy(float)
    n = len(electrodes)
    rows = []
    if n < 2:
        return pd.DataFrame(columns=["ref_idx", "target_idx", "ref_electrode", "target_electrode", "distance_um", "distance_bin_um"])
    tri_i, tri_j = np.triu_indices(n, k=1)
    d = np.sqrt(np.sum((coords_sel[tri_i] - coords_sel[tri_j]) ** 2, axis=1))
    edges = np.arange(LYAP_MIN_DISTANCE_UM, LYAP_MAX_DISTANCE_UM + LYAP_DISTANCE_BIN_UM, LYAP_DISTANCE_BIN_UM)
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_idx = np.digitize(d, edges) - 1
    for b, center in enumerate(centers):
        idx = np.where(bin_idx == b)[0]
        if idx.size == 0:
            continue
        if idx.size > LYAP_MAX_PAIRS_PER_DISTANCE_BIN:
            idx = rng.choice(idx, size=LYAP_MAX_PAIRS_PER_DISTANCE_BIN, replace=False)
        for k in idx:
            rows.append({
                "ref_idx": int(tri_i[k]),
                "target_idx": int(tri_j[k]),
                "ref_electrode": int(electrodes[tri_i[k]]),
                "target_electrode": int(electrodes[tri_j[k]]),
                "distance_um": float(d[k]),
                "distance_bin_um": float(center),
            })
    return pd.DataFrame(rows)


def analyze_recording_lyapunov(activity, seed=RANDOM_SEED):
    spec = activity["spec"]
    if not activity["ok"]:
        return pd.DataFrame(), {
            "dataset": spec["dataset"], "well": spec["well"], "div": spec["div"],
            "recording_id": spec["recording_id"], "valid": False, "skip_reason": activity["skip_reason"],
            "n_selected_electrodes": 0, "selected_duration_s": 0.0, "n_pair_samples": 0,
        }
    selected_intervals = activity["selected_intervals"]
    if len(selected_intervals) == 0:
        return pd.DataFrame(), {
            "dataset": spec["dataset"], "well": spec["well"], "div": spec["div"],
            "recording_id": spec["recording_id"], "valid": False, "skip_reason": "no selected intervals",
            "n_selected_electrodes": int(len(activity["refs"])), "selected_duration_s": 0.0, "n_pair_samples": 0,
        }
    population = activity["population"]
    rec = activity["recording"]
    refs = np.asarray(activity["refs"], dtype=int)
    t = np.asarray(population.time_grid, dtype=float)
    mask = interval_mask(t, selected_intervals)
    if np.count_nonzero(mask) < (LYAP_EMBED_DIM * LYAP_DELAY_BINS + LYAP_MAX_DT_BINS + 5):
        return pd.DataFrame(), {
            "dataset": spec["dataset"], "well": spec["well"], "div": spec["div"],
            "recording_id": spec["recording_id"], "valid": False, "skip_reason": "selected intervals too short",
            "n_selected_electrodes": int(len(refs)), "selected_duration_s": float(np.sum(np.diff(selected_intervals, axis=1))), "n_pair_samples": 0,
        }
    coords = electrode_coords(rec)
    refs = np.asarray([e for e in refs if int(e) in coords.index], dtype=int)
    rng = np.random.default_rng(seed + int(spec["div"]) * 100 + int(spec["well"]))
    pairs = sample_pairs_by_distance(refs, coords, rng)
    if pairs.empty:
        return pd.DataFrame(), {
            "dataset": spec["dataset"], "well": spec["well"], "div": spec["div"],
            "recording_id": spec["recording_id"], "valid": False, "skip_reason": "no sampled pairs",
            "n_selected_electrodes": int(len(refs)), "selected_duration_s": float(np.sum(np.diff(selected_intervals, axis=1))), "n_pair_samples": 0,
        }
    rates = build_rate_matrix(rec, refs, t)
    rates = rates[:, mask]
    embedding = make_delay_embedding_matrix(rates)
    rows = []
    for row in pairs.itertuples(index=False):
        lyap = compute_pair_lyapunov_from_embeddings(embedding[:, row.ref_idx, :], embedding[:, row.target_idx, :], rng)
        if np.isfinite(lyap):
            rows.append({
                "dataset": spec["dataset"],
                "well": int(spec["well"]),
                "div": int(spec["div"]),
                "recording_id": spec["recording_id"],
                "ref_electrode": int(row.ref_electrode),
                "target_electrode": int(row.target_electrode),
                "distance_um": float(row.distance_um),
                "distance_bin_um": float(row.distance_bin_um),
                "lyapunov": float(lyap),
            })
    pair_df = pd.DataFrame(rows)
    summary = {
        "dataset": spec["dataset"],
        "well": int(spec["well"]),
        "div": int(spec["div"]),
        "recording_id": spec["recording_id"],
        "valid": bool(not pair_df.empty),
        "skip_reason": "" if not pair_df.empty else "no finite pair lyapunov values",
        "n_selected_electrodes": int(len(refs)),
        "n_high_activity_intervals": int(len(activity["high_epochs"])),
        "n_burst_intervals": int(len(activity["burst_epochs"])),
        "n_selected_intervals": int(len(selected_intervals)),
        "selected_duration_s": float(np.sum(selected_intervals[:, 1] - selected_intervals[:, 0])),
        "n_pair_samples": int(len(pair_df)),
    }
    return pair_df, summary

## Per-DIV Build or Load

Each DIV writes three compact tables: pair samples, distance summaries, and recording summaries. This is the cell to rerun when changing analysis parameters.

In [ ]:
def summarize_distance_bins(pair_df):
    if pair_df.empty:
        return pd.DataFrame(columns=["div", "distance_bin_um", "mean_lyapunov", "median_lyapunov", "ci_low", "ci_high", "n_pairs", "n_recordings"])
    rows = []
    rng = np.random.default_rng(RANDOM_SEED + 991)
    for (div, dist), group in pair_df.groupby(["div", "distance_bin_um"], sort=True):
        values = group["lyapunov"].to_numpy(float)
        rec_ids = group["recording_id"].dropna().unique()
        if len(rec_ids) > 1 and LYAP_BOOTSTRAP_REPS > 0:
            boot = []
            by_rec = {rid: group.loc[group["recording_id"] == rid, "lyapunov"].to_numpy(float) for rid in rec_ids}
            for _ in range(LYAP_BOOTSTRAP_REPS):
                sample_recs = rng.choice(rec_ids, size=len(rec_ids), replace=True)
                vals = np.concatenate([by_rec[rid] for rid in sample_recs if by_rec[rid].size])
                if vals.size:
                    boot.append(np.nanmean(vals))
            ci_low, ci_high = np.nanpercentile(boot, [2.5, 97.5]) if boot else (np.nan, np.nan)
        elif values.size > 1:
            se = np.nanstd(values, ddof=1) / np.sqrt(values.size)
            ci_low = np.nanmean(values) - 1.96 * se
            ci_high = np.nanmean(values) + 1.96 * se
        else:
            ci_low = ci_high = np.nan
        rows.append({
            "div": int(div),
            "distance_bin_um": float(dist),
            "mean_lyapunov": float(np.nanmean(values)),
            "median_lyapunov": float(np.nanmedian(values)),
            "ci_low": float(ci_low),
            "ci_high": float(ci_high),
            "n_pairs": int(values.size),
            "n_recordings": int(len(rec_ids)),
        })
    return pd.DataFrame(rows)


def build_or_load_div(div):
    paths = div_cache_paths(div)
    if int(div) not in set(map(int, FORCE_RECOMPUTE_DIVS)) and all(paths[k].exists() for k in ("summary", "pairs", "recordings", "payload")):
        print(f"DIV {div}: loading compatible Lyapunov cache")
        return {
            "distance_summary": pd.read_csv(paths["summary"]),
            "pair_samples": pd.read_csv(paths["pairs"]),
            "recording_summary": pd.read_csv(paths["recordings"]),
        }
    print(f"DIV {div}: building Lyapunov tables")
    t0 = time.time()
    pair_parts = []
    rec_rows = []
    for spec in iter_recording_specs_for_div(div):
        print(f"  {spec['recording_id']}")
        activity = detect_recording_activity(spec)
        pair_df, rec_summary = analyze_recording_lyapunov(activity)
        rec_rows.append(rec_summary)
        if not pair_df.empty:
            pair_parts.append(pair_df)
    pair_df = pd.concat(pair_parts, ignore_index=True) if pair_parts else pd.DataFrame()
    rec_df = pd.DataFrame(rec_rows)
    dist_df = summarize_distance_bins(pair_df)
    pair_df.to_csv(paths["pairs"], index=False, compression="gzip")
    rec_df.to_csv(paths["recordings"], index=False, compression="gzip")
    dist_df.to_csv(paths["summary"], index=False, compression="gzip")
    with paths["payload"].open("wb") as f:
        pickle.dump({"config": lyap_config_dict(), "div": int(div)}, f)
    print(f"DIV {div}: wrote {len(pair_df)} pair rows in {(time.time() - t0) / 60:.1f} min")
    return {"distance_summary": dist_df, "pair_samples": pair_df, "recording_summary": rec_df}


div_payloads = {int(div): build_or_load_div(div) for div in REQUESTED_DIVS}
lyap_distance_summary = pd.concat([p["distance_summary"] for p in div_payloads.values()], ignore_index=True)
lyap_pair_samples = pd.concat([p["pair_samples"] for p in div_payloads.values()], ignore_index=True)
lyap_recording_summary = pd.concat([p["recording_summary"] for p in div_payloads.values()], ignore_index=True)

lyap_distance_summary.to_csv(TABLE_DIR / f"lyapunov_distance_summary_{CONFIG_DIGEST}.csv", index=False)
lyap_recording_summary.to_csv(TABLE_DIR / f"lyapunov_recording_summary_{CONFIG_DIGEST}.csv", index=False)

print("distance summary", lyap_distance_summary.shape)
print("pair samples", lyap_pair_samples.shape)
print("recording summary", lyap_recording_summary.shape)
lyap_recording_summary

## Inefficiency Review

This table is meant to be kept with the figure notebook so we can see which knobs control runtime before scaling up pair counts.

In [ ]:
inefficiency_notes = pd.DataFrame([
    {
        "component": "legacy pair loop",
        "risk": "delay vectors recomputed for every electrode pair",
        "mitigation_here": "delay embeddings are created once per recording/electrode",
    },
    {
        "component": "nearest-neighbor divergence",
        "risk": "O(T^2) distance matrix per sampled pair",
        "mitigation_here": "sample at most LYAP_MAX_TIME_POINTS time points per pair",
    },
    {
        "component": "all-pairs scan",
        "risk": "~N^2 pairs per recording can exceed hundreds of thousands",
        "mitigation_here": "sample at most LYAP_MAX_PAIRS_PER_DISTANCE_BIN pairs per distance bin",
    },
    {
        "component": "activity-state preprocessing",
        "risk": "high-resolution traces are costly when rebuilt repeatedly",
        "mitigation_here": "one state detection pass per recording; per-DIV caches preserve results",
    },
    {
        "component": "cache size",
        "risk": "raw traces or embeddings create large files",
        "mitigation_here": "only pair samples, distance summaries, recording summaries, and a small config marker are written",
    },
])
inefficiency_notes

## Figure 6 Panels

In [ ]:
DIV_COLORS = {int(div): plt.cm.viridis(i / max(1, len(REQUESTED_DIVS) - 1)) for i, div in enumerate(REQUESTED_DIVS)}


def draw_distance_lyapunov_panel(ax):
    if lyap_distance_summary.empty:
        ax.text(0.5, 0.5, "No Lyapunov summaries", ha="center", va="center", transform=ax.transAxes)
        return
    for div in REQUESTED_DIVS:
        df = lyap_distance_summary.loc[lyap_distance_summary["div"].astype(int) == int(div)].sort_values("distance_bin_um")
        if df.empty:
            continue
        x = df["distance_bin_um"].to_numpy(float)
        y = df["mean_lyapunov"].to_numpy(float)
        ax.plot(x, y, marker="o", ms=2.2, lw=0.9, color=DIV_COLORS[int(div)], label=f"DIV {div}")
        if {"ci_low", "ci_high"}.issubset(df.columns):
            lo = df["ci_low"].to_numpy(float)
            hi = df["ci_high"].to_numpy(float)
            ok = np.isfinite(lo) & np.isfinite(hi)
            if np.any(ok):
                ax.fill_between(x[ok], lo[ok], hi[ok], color=DIV_COLORS[int(div)], alpha=0.12, linewidth=0)
    ax.axhline(0, color="0.65", lw=0.5)
    ax.set_xlabel("distance (um)")
    ax.set_ylabel("Lyapunov-style divergence")
    ax.set_title("Distance-dependent stability")
    ax.legend(frameon=False, ncol=2)


def draw_recording_support_panel(ax):
    if lyap_recording_summary.empty:
        ax.text(0.5, 0.5, "No recording summary", ha="center", va="center", transform=ax.transAxes)
        return
    df = lyap_recording_summary.copy()
    df["div"] = df["div"].astype(int)
    for div in REQUESTED_DIVS:
        sub = df.loc[df["div"] == int(div)]
        if sub.empty:
            continue
        x = np.full(len(sub), int(div), dtype=float) + np.linspace(-0.13, 0.13, len(sub))
        ax.scatter(x, sub["n_pair_samples"], color=DIV_COLORS[int(div)], s=12, alpha=0.85)
        valid = pd.to_numeric(sub["valid"], errors="coerce").fillna(False).astype(bool)
        ax.text(int(div), max(1, sub["n_pair_samples"].max()) * 1.03, f"{valid.sum()}/{len(sub)}", ha="center", va="bottom")
    ax.set_xticks(REQUESTED_DIVS)
    ax.set_xlabel("DIV")
    ax.set_ylabel("finite pair samples")
    ax.set_title("Per-recording support")


def draw_lyapunov_distribution_panel(ax):
    if lyap_pair_samples.empty:
        ax.text(0.5, 0.5, "No pair samples", ha="center", va="center", transform=ax.transAxes)
        return
    bins = np.linspace(np.nanpercentile(lyap_pair_samples["lyapunov"], 1), np.nanpercentile(lyap_pair_samples["lyapunov"], 99), 45)
    for div in REQUESTED_DIVS:
        vals = lyap_pair_samples.loc[lyap_pair_samples["div"].astype(int) == int(div), "lyapunov"].to_numpy(float)
        vals = vals[np.isfinite(vals)]
        if vals.size:
            ax.hist(vals, bins=bins, histtype="step", density=True, lw=0.9, color=DIV_COLORS[int(div)], label=f"DIV {div}")
    ax.axvline(0, color="0.6", lw=0.5)
    ax.set_xlabel("pair Lyapunov-style divergence")
    ax.set_ylabel("density")
    ax.set_title("Pair-level distributions")


def draw_div_heatmap_panel(ax):
    if lyap_distance_summary.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        return
    pivot = lyap_distance_summary.pivot_table(index="div", columns="distance_bin_um", values="mean_lyapunov", aggfunc="mean")
    pivot = pivot.reindex(index=REQUESTED_DIVS)
    im = ax.imshow(pivot.to_numpy(float), aspect="auto", origin="lower", cmap="magma")
    ax.set_yticks(np.arange(len(pivot.index)), [str(int(v)) for v in pivot.index])
    col_vals = np.asarray(pivot.columns, dtype=float)
    tick_idx = np.linspace(0, len(col_vals) - 1, min(5, len(col_vals)), dtype=int) if len(col_vals) else []
    ax.set_xticks(tick_idx, [f"{col_vals[i]:.0f}" for i in tick_idx])
    ax.set_xlabel("distance (um)")
    ax.set_ylabel("DIV")
    ax.set_title("Mean divergence heatmap")
    plt.colorbar(im, ax=ax, label="divergence", fraction=0.046, pad=0.02)


def compose_figure6():
    return compose_figure(
        figure(width="two_column", height=115, grid=(2, 12), title="Lyapunov-style spike-rate stability by DIV"),
        [
            panel("a", label="a", loc=(0, 0, 1, 7), draw=lambda data, ax, **opts: draw_distance_lyapunov_panel(ax)),
            panel("b", label="b", loc=(0, 7, 1, 5), draw=lambda data, ax, **opts: draw_div_heatmap_panel(ax)),
            panel("c", label="c", loc=(1, 0, 1, 5), draw=lambda data, ax, **opts: draw_lyapunov_distribution_panel(ax)),
            panel("d", label="d", loc=(1, 5, 1, 7), draw=lambda data, ax, **opts: draw_recording_support_panel(ax)),
        ],
    )


fig6 = compose_figure6()
plt.show()

## Export

In [ ]:
if SAVE_FIGURE:
    export_figure(fig6.fig, OUTPUT_DIR / "figure6_lyapunov_compilation", formats=("pdf", "svg", "png"))
    print(f"Saved Figure 6 draft to {OUTPUT_DIR}")